<a href="https://colab.research.google.com/github/KinhVan2x1/AIO_2024_Excercise/blob/feature/Week4_homework_AIO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 1 STREAMLIT #

In [ ]:
# Install Streamlit
!pip install -q streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 19.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import streamlit as st

# Load file txt with vocabs
def load_vocab(file_path):
  with open(file_path, 'r') as f:
    lines = f.readlines()
  words = sorted(set([line.strip().lower() for line in lines]))
  return words

vocabs = load_vocab(file_path='./vocab.txt')

# Levenshtein_distance
def levenshtein_distance(token1, token2):
  distances = [[0]*(len(token2) + 1) for i in range(len(token1) + 1)]

  for t1 in range(len(token1)+1):
    distances[t1][0] = t1

  for t2 in range(len(token2)+1):
    distances[0][t2] = t2

  a = 0
  b = 0
  c = 0

  for t1 in range(1, len(token1) + 1):
    for t2 in range(1, len(token2) + 1):
      if (token1[t1 - 1] == token2[t2 - 1]):
        distances[t1][t2] = distances[t1 -1][t2 -1]
      else:
          a = distances[t1][t2 - 1]
          b = distances[t1 - 1][t2]
          c = distances[t1 - 1][t2 - 1]

          if (a <=  b and a <= c):
            distances[t1][t2] = a + 1
          elif (b <= a and b <= c):
            distances[t1][t2] = b + 1
          else:
            distances[t1][t2] = c + 1

  return distances[len(token1)][len(token2)]

# Main
def main():
  st.title('Word Correction using Levenshtein Distance')
  word = st.text_input('Word: ')

  if st.button('Compute'):

    # Compute levnshtein distance
    leven_distance = dict()
    for vocab in vocabs:
      leven_distance[vocab] = levenshtein_distance(word, vocab)

    # Sorted by distance
    sorted_distances = dict(sorted(leven_distance.items(), key = lambda item: item[1]))
    correct_word = list(sorted_distances.keys())[0]
    st.write(f'Correct word: {correct_word}')

    col1, col2 = st.columns(2)
    col1.write('Vocabulary')
    col1.write(vocabs)

    col2.write('Distances:')
    col2.write(sorted_distances)

if __name__ == '__main__':
  main()

Writing app.py


In [ ]:
!pip install pyngrok
from pyngrok import ngrok
!ngrok config add-authtoken 2ifhXUjwNwjgZbBbNzbcG8NVNDa_35PoEujHR45u7GkvoZu9a

# Start a new ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f'Streamlit app running at: {public_url}')

# Run the Streamlit app
!streamlit run app.py --server.port 8000

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Streamlit app running at: https://6ac8-35-236-168-122.ngrok-free.app



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8000
  Network URL: http://172.28.0.12:8000
  External URL: http://35.236.168.122:8000

2024-07-03 04:38:39.008 Uncaught app exception
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/streamlit/runtime/scriptrunner/script_runner.py", line 589, in _run_script
    exec(code, module.__dict__)
  File "/content/app.py", line 10, in <module>
    vocabs = load_vocab(file_path='./vocab.txt')
  File "/content/app.py", line 5, in load_vocab
    with open(file_path, 'r') as f:
FileNotFoundError: [Errno 2] No such file or directory: './vocab.txt'
  Stopping...
^C


# EXCERCISE 2 _ OBJECT DETECTION #

In [ ]:
from google.colab import files
upload = files.upload()

In [ ]:
%%writefile app.py
import cv2
import numpy as np
from PIL import Image
import streamlit as st

model = 'model/MobileNetSSD_deploy.caffemodel'
prototxt = 'model/MobileNetSSD_deploy.prototxt.txt'

def process_image(image):
  blob = cv2.dnn.blobFromImage(
      cv2.resize(image, (300, 300)), 0.007843, (300, 300), 127.5
  )
  net = cv2.dnn.readNet(model, prototxt)
  net.setInput(blob)
  detections = net.forward()
  return detections

def annotate_image(image, detections, confidence_threshold = 0.5):
  # Loop over the detections:
  (h, w) = image.shape[:2]
  for i in np.arange(0, detections.shape[2]):
    confidence = detections[0, 0, i, 2]
    if confidence > confidence_threshold:
      idx = int(detections[0, 0, i, 1])
      box = detections[0, 0, i, 3:7] * np.array([w, h, w, h])
      (startX, startY, endX, endY) = box.astype("int")
      cv2.rectangle(image, (startX, startY), (endX, endY), 70, 2)
  return image

def main():
  st.title('Object Detection for Images')
  file = st.file_uploader('Upload Image', type=['jpg', 'png'])
  if file is not None:
    image = Image.open(file)
    image = np.array(image)
    detections = process_image(image)
    processed_image = annotate_image(image, detections)
    st.image(processed_image, caption = 'Processed Image')

    st.image(image, caption='Uploaded Image', use_column_width=True)

if __name__ == '__main__':
  main()

Overwriting app.py


In [ ]:
!pip install pyngrok
from pyngrok import ngrok
!ngrok config add-authtoken 2ifhXUjwNwjgZbBbNzbcG8NVNDa_35PoEujHR45u7GkvoZu9a

# Start a new ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f'Streamlit app running at: {public_url}')

# Run the Streamlit app
!streamlit run app.py --server.port 8000

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Streamlit app running at: https://e7e7-35-236-168-122.ngrok-free.app



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8000
  Network URL: http://172.28.0.12:8000
  External URL: http://35.236.168.122:8000



# EXERCISE 3 CHATBOT


In [ ]:
!pip install-q hugchat

In [ ]:
%%writefile app.py
import streamlit as st
from hugchat import hugchat
from hugchat.login import Login

# App title
st.title('Simple ChatBot')

# Hugging Face Credentials
with st.sidebar:
  st.title('Login Hugchat')
  hf_email = st.text_input('Enter Email:')
  hf_pass = st.text_input('Enter password:', type='password')
  if not (hf_email and hf_pass):
    st.warning('Please enter your account!')
  else:
    st.success('Proceed to entering your prompt message')

# Store LLM generated responses
if 'messages' not in st.session_state.keys():
  st.session_state.messages = [{'role': 'assisstant',
                               'content': 'How may I help you?'}]

# Display chat message
for message in st.session_state.messages:
  with st.chat_message(message['role']):
    st.write(message['content'])

# Function for generating LLM response
def generate_response(prompt_input, email, passwd):
    # Hugging Face Login
    sign = Login(email, passwd)
    cookies = sign.login()
    # Create ChatBot
    chatbot = hugchat.ChatBot(cookies=cookies.get_dict())
    return chatbot.chat(prompt_input)

# User-provided prompt
if prompt := st.chat_input(disabled= not (hf_email and hf_pass)):
  st.session_state.messages.append({'role': 'user',
                                    'content': prompt})
  with st.chat_message('user'):
    st.write(prompt)

# Generate a new response if last message is not from assistant
if st.session_state.messages[-1]["role"] != "assistant":
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = generate_response(prompt, hf_email, hf_pass)
            st.write(response)
    message = {"role": "assistant", "content": response}
    st.session_state.messages.append(message)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
!ngrok config add-authtoken 2ifhXUjwNwjgZbBbNzbcG8NVNDa_35PoEujHR45u7GkvoZu9a

# Start a new ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f'Streamlit app running at: {public_url}')

# Run the Streamlit app
!streamlit run app.py --server.port 8000

In [ ]:
%%writefile app.py
import streamlit as st
from hugchat import hugchat
from hugchat.login import Login

# App title
st.title('Simple ChatBot')

# Hugging Face Credentials
with st.sidebar:
    st.title('Login HugChat')
    hf_email = st.text_input('Enter E-mail:')
    hf_pass = st.text_input('Enter Password:', type='password')
    if not (hf_email and hf_pass):
        st.warning('Please enter your account!', icon='⚠️')
    else:
        st.success('Proceed to entering your prompt message!', icon='👉')

# Store LLM generated responses
if "messages" not in st.session_state.keys():
    st.session_state.messages = [{"role": "assistant", "content": "How may I help you?"}]

# Display chat messages
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.write(message["content"])

# Function for generating LLM response
def generate_response(prompt_input, email, passwd):
    # Hugging Face Login
    sign = Login(email, passwd)
    cookies = sign.login()
    # Create ChatBot
    chatbot = hugchat.ChatBot(cookies=cookies.get_dict())
    return chatbot.chat(prompt_input)

# User-provided prompt
if prompt := st.chat_input(disabled=not (hf_email and hf_pass)):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

# Generate a new response if last message is not from assistant
if st.session_state.messages[-1]["role"] != "assistant":
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = generate_response(prompt, hf_email, hf_pass)
            st.write(response)
    message = {"role": "assistant", "content": response}
    st.session_state.messages.append(message)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
!ngrok config add-authtoken 2ifhXUjwNwjgZbBbNzbcG8NVNDa_35PoEujHR45u7GkvoZu9a

# Start a new ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f'Streamlit app running at: {public_url}')

# Run the Streamlit app
!streamlit run app.py --server.port 8000